# Richard Texan oil planning

Springfield's Richard Texan sells three types of gasoline made by blending three types of crude oil. Each crude has a purchase price, an octane level, and a sulfur percentage:

| Crude | Price (dollars/barrel) | Octane | Sulfur (%) |
| --- | --- | --- | --- |
| 1 | 45 | 12 | 0.5 |
| 2 | 35 | 6 | 2.0 |
| 3 | 25 | 8 | 3.0 |

Each gasoline has a selling price, a minimum octane level, a maximum sulfur percentage, and a nominal daily demand:

| Gasoline | Price (dollars/barrel) | Minimum octane | Maximum sulfur (%) | Nominal demand (barrels/day) |
| --- | --- | --- | --- | --- |
| 1 | 70 | 10 | 1.0 | 3000 |
| 2 | 60 | 8 | 2.0 | 2000 |
| 3 | 50 | 6 | 1.0 | 1000 |

Nominal demand is the **maximum** quantity that can be sold without advertising. Each dollar spent advertising a particular gasoline increases its demand limit by 10 barrels.

Richard can purchase at most 5000 barrels of each crude per day and process at most 14000 barrels total per day. Processing costs 4 dollars per barrel. Determine the purchases, blends, sales, and advertising that maximize daily profit.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. JuMP, HiGHS, and Printf are already included. No external data files are needed.

This classroom model allows fractional barrels, assumes no volume loss or inventory, and treats octane and sulfur as weighted averages of the crude inputs.

## Problem data

Dictionaries associate each crude or gasoline symbol with its data. Sulfur values use the percentage scale: `0.5` means 0.5%. All barrel quantities and advertising expenditures refer to one day.

`processing_cost_per_barrel` is a numerical input. We will use a different name, `processing_expense`, for the model expression that computes the total processing cost.

In [ ]:
using JuMP, HiGHS, Printf
import MathOptInterface as MOI

gases = [:gas1, :gas2, :gas3]
crudes = [:crude1, :crude2, :crude3]

min_octane = Dict(zip(gases, [10, 8, 6]))
max_sulfur = Dict(zip(gases, [1, 2, 1]))
gas_price = Dict(zip(gases, [70, 60, 50]))
gas_nom_demand = Dict(zip(gases, [3000, 2000, 1000]))

crude_price = Dict(zip(crudes, [45, 35, 25]))
octane = Dict(zip(crudes, [12, 6, 8]))
sulfur = Dict(zip(crudes, [0.5, 2.0, 3.0]))

max_crude_available = 5000
max_crude_processed = 14000
processing_cost_per_barrel = 4
advertising_inc = 10  # Extra barrels of demand per advertising dollar.

## Decisions and material balances

Use four sets of nonnegative variables:

| Variable | Meaning | Units |
| --- | --- | --- |
| $x_{ij}$ | Crude $i$ blended into gasoline $j$ | Barrels/day |
| $y_i$ | Crude $i$ purchased and processed | Barrels/day |
| $z_j$ | Gasoline $j$ produced and sold | Barrels/day |
| $a_j$ | Advertising expenditure for gasoline $j$ | Dollars/day |

The blend allocations must account for every purchased barrel and every sold barrel:

$$
\sum_j x_{ij} = y_i \quad \forall i,
\qquad
\sum_i x_{ij} = z_j \quad \forall j.
$$

These equalities connect purchasing, blending, and sales. The separate $y$ and $z$ variables make the remaining expressions easier to read.

In [ ]:
model = Model(HiGHS.Optimizer)
set_silent(model)  # Remove this line to see the solver log.

@variable(model, y[crudes] >= 0)
@variable(model, z[gases] >= 0)
@variable(model, x[crudes, gases] >= 0)
@variable(model, a[gases] >= 0)

@constraint(model, crude_balance[i in crudes], sum(x[i, j] for j in gases) == y[i])
@constraint(model, gas_balance[j in gases], sum(x[i, j] for i in crudes) == z[j])

## Maximize daily profit

Profit is sales revenue minus crude purchases, processing, and advertising:

$$
\max \left(
\sum_j p_j z_j
- \sum_i c_i y_i
- k \sum_i y_i
- \sum_j a_j
\right),
$$

where $p_j$ is the gasoline selling price, $c_i$ is the crude purchase price, and $k$ is the processing charge per barrel. Every term is measured in dollars per day.

Named JuMP expressions let us build the objective in parts and report those same parts after solving.

In [ ]:
@expression(model, revenue, sum(gas_price[j] * z[j] for j in gases))
@expression(model, crude_expense, sum(crude_price[i] * y[i] for i in crudes))
@expression(model, processing_expense, processing_cost_per_barrel * sum(y[i] for i in crudes))
@expression(model, advertising_expense, sum(a[j] for j in gases))
@objective(model, Max, revenue - crude_expense - processing_expense - advertising_expense)

## Capacity, demand, and quality constraints

Purchases of each crude cannot exceed 5000 barrels, and total processing cannot exceed 14000 barrels. Sales of gasoline $j$ satisfy

$$
z_j \leq D_j + 10a_j,
$$

where $D_j$ is its nominal demand. There is no minimum sales requirement. Advertising raises the sales limit for its own gasoline, and its dollar cost is deducted from profit.

For positive production $z_j$, the average octane is $\sum_i o_i x_{ij}/z_j$. Multiplying by $z_j$ gives the linear quality constraints

$$
\sum_i o_i x_{ij} \geq O_j z_j,
\qquad
\sum_i s_i x_{ij} \leq S_j z_j,
$$

where $o_i$ and $s_i$ are crude octane and sulfur, and $O_j$ and $S_j$ are the gasoline limits. Sulfur inputs and limits are both percentages, so their factors of $1/100$ cancel.

When $z_j = 0$, its balance equation and nonnegative blend variables force every $x_{ij}$ to be zero. Both quality constraints then reduce to $0 \geq 0$ or $0 \leq 0$. The LP therefore needs no division and allows a gasoline to go unproduced.

In [ ]:
@constraint(model, crude_limit[i in crudes], y[i] <= max_crude_available)
@constraint(model, processing_limit, sum(y[i] for i in crudes) <= max_crude_processed)
@constraint(model, demand_limit[j in gases], z[j] <= gas_nom_demand[j] + advertising_inc * a[j])

@constraint(model, octane_limit[j in gases],
    sum(octane[i] * x[i, j] for i in crudes) >= min_octane[j] * z[j])
@constraint(model, sulfur_limit[j in gases],
    sum(sulfur[i] * x[i, j] for i in crudes) <= max_sulfur[j] * z[j])

model

## Solve and report the operating plan

Check the solver status and the availability of a feasible optimal solution before reading values. First report the profit calculation, then inspect crude purchases, gasoline sales, and advertising.

The reports use rounded numbers for readability; all calculations use the unrounded solution.

In [ ]:
optimize!(model)
status = termination_status(model)
println("Termination status: ", status)
status == MOI.OPTIMAL || error("HiGHS stopped with status $(status).")
is_solved_and_feasible(model) || error("No feasible optimal solution is available.")

maximum_profit = objective_value(model)
daily_revenue = value(revenue)
daily_crude_cost = value(crude_expense)
daily_processing_cost = value(processing_expense)
daily_advertising_cost = value(advertising_expense)
total_processed = sum(value(y[i]) for i in crudes)

@printf("\nMaximum daily profit: \$%.2f\n", maximum_profit)
@printf("Revenue: \$%.2f\n", daily_revenue)
@printf("Crude purchases: \$%.2f\n", daily_crude_cost)
@printf("Processing: \$%.2f\n", daily_processing_cost)
@printf("Advertising: \$%.2f\n", daily_advertising_cost)

@printf("\nTotal processed: %.2f of %.2f barrels/day\n",
    total_processed, max_crude_processed)
for i in crudes
    @printf("Buy %.2f barrels of %s (limit %.2f)\n",
        value(y[i]), i, max_crude_available)
end

println("\nGasoline sales and advertising:")
for j in gases
    demand_with_ads = gas_nom_demand[j] + advertising_inc * value(a[j])
    @printf("%s: sell %.2f barrels; advertise \$%.2f; sales limit %.2f barrels\n",
        j, value(z[j]), value(a[j]), demand_with_ads)
end

## Check each blend

For each gasoline produced, display its crude allocations and recompute its average octane and sulfur percentage. A gasoline with zero production has no mixture to average, so report that case without dividing by zero.

In [ ]:
# Omit averages for products with no production (within numerical tolerance).
produced_gases = [j for j in gases if value(z[j]) > 1e-6]
blend_octane = Dict(
    j => sum(octane[i] * value(x[i, j]) for i in crudes) / value(z[j])
    for j in produced_gases
)
blend_sulfur = Dict(
    j => sum(sulfur[i] * value(x[i, j]) for i in crudes) / value(z[j])
    for j in produced_gases
)

for j in gases
    if j in produced_gases
        println("\nBlend for ", j, ":")
        for i in crudes
            @printf("  %s: %.4f barrels\n", i, value(x[i, j]))
        end
        @printf("  Average octane: %.4f (minimum %.2f)\n",
            blend_octane[j], min_octane[j])
        @printf("  Average sulfur: %.4f%% (maximum %.2f%%)\n",
            blend_sulfur[j], max_sulfur[j])
    else
        println("\n", j, ": no production; average quality is not defined.")
    end
end

## Interpret the plan

With the supplied data, the maximum daily profit is **318,100 dollars**.

Which gasolines receive advertising? Which crude supplies, processing capacity, demand limits, and quality limits are binding? Why might the optimal plan leave a gasoline unproduced even when its selling price exceeds some crude purchase prices?

Compare this model with `06-Alloy.ipynb`. Both constrain the quality of a blend, but this model chooses several blends at once and also chooses advertising expenditures to expand sales limits.

To explore a different processing charge, change `processing_cost_per_barrel` in the data cell and choose **Run All**. Save a personal copy in `student-work/` if you want to keep your edits.